# Experiment: Life Threshold with GRA obnulënka

This notebook simulates a system transitioning from non‑life to proto‑life to life. GRA frames compete to correctly classify the state, and the reset mechanism adapts the population as the system evolves.

## 1. Imports and setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from src.core import GRAResetController, GRAAgent
from src.domains.bio_life import NonLivingFrame, ProtoLifeFrame, LifeFrame

%matplotlib inline

## 2. Generate synthetic data for three regimes

We create three datasets representing different stages of a living system:

- **Non‑living**: life score ~ 0.1 (low)
- **Proto‑life**: life score ~ 0.5 (medium)
- **Life**: life score ~ 0.95 (high)

Each dataset contains 10 measurements with small noise.


In [ ]:
np.random.seed(42)

n_points = 10

non_living_data = 0.1 + 0.02 * np.random.randn(n_points)
proto_life_data = 0.5 + 0.05 * np.random.randn(n_points)
life_data = 0.95 + 0.02 * np.random.randn(n_points)

# Clip to [0,1]
non_living_data = np.clip(non_living_data, 0, 1)
proto_life_data = np.clip(proto_life_data, 0, 1)
life_data = np.clip(life_data, 0, 1)

print("Non‑living data mean: {:.3f}".format(non_living_data.mean()))
print("Proto‑life data mean: {:.3f}".format(proto_life_data.mean()))
print("Life data mean: {:.3f}".format(life_data.mean()))

## 3. Create initial population of frames

We create 5 frames of each type with random thresholds.


In [ ]:
def make_initial_population():
    frames = []
    # NonLivingFrame: threshold above which it's wrong (default 0.2)
    for i in range(5):
        th = np.random.uniform(0.1, 0.3)
        f = NonLivingFrame(f"nonliving_{i}")
        f.instantiate(threshold=th)
        frames.append(f)
    # ProtoLifeFrame: low and high bounds
    for i in range(5):
        low = np.random.uniform(0.2, 0.4)
        high = np.random.uniform(0.6, 0.8)
        f = ProtoLifeFrame(f"protolife_{i}")
        f.instantiate(low=low, high=high)
        frames.append(f)
    # LifeFrame: minimum life score
    for i in range(5):
        min_life = np.random.uniform(0.8, 0.95)
        f = LifeFrame(f"life_{i}")
        f.instantiate(min_life=min_life)
        frames.append(f)
    return frames

frames = make_initial_population()
print(f"Created {len(frames)} frames.")

## 4. Define reset policy and mutation agent

We keep the top 30% of frames based on their score (lower is better). Then we generate children by mutating the parameters of survivors.


In [ ]:
def top_k_policy(scores, frames, k=0.3):
    sorted_frames = sorted(frames, key=lambda f: scores[f.name])
    n_keep = int(len(frames) * k)
    survivors = sorted_frames[:n_keep]
    killed = sorted_frames[n_keep:]
    return survivors, killed

def mutate_frame(frame):
    """Create a child frame by slightly perturbing parameters."""
    if isinstance(frame, NonLivingFrame):
        new_th = frame.params['threshold'] * np.random.uniform(0.9, 1.1)
        new_th = np.clip(new_th, 0.05, 0.5)
        child = NonLivingFrame(frame.name+"_child", parent=frame)
        child.instantiate(threshold=new_th)
    elif isinstance(frame, ProtoLifeFrame):
        low = frame.params['low'] * np.random.uniform(0.9, 1.1)
        high = frame.params['high'] * np.random.uniform(0.9, 1.1)
        # ensure low < high
        if low > high:
            low, high = high, low
        low = np.clip(low, 0.1, 0.9)
        high = np.clip(high, 0.2, 1.0)
        child = ProtoLifeFrame(frame.name+"_child", parent=frame)
        child.instantiate(low=low, high=high)
    elif isinstance(frame, LifeFrame):
        new_min = frame.params['min_life'] * np.random.uniform(0.9, 1.1)
        new_min = np.clip(new_min, 0.7, 0.99)
        child = LifeFrame(frame.name+"_child", parent=frame)
        child.instantiate(min_life=new_min)
    else:
        child = frame
    return child

agent = GRAAgent([mutate_frame])
controller = GRAResetController(frames, lambda s,f: top_k_policy(s,f,k=0.3))

## 5. Run the reset cycle while the system evolves

We simulate 30 epochs. In each epoch we feed one of the three datasets in sequence: first non‑living, then proto‑life, then life, and repeat. This mimics the gradual emergence of life.


In [ ]:
n_epochs = 30
datasets = [non_living_data, proto_life_data, life_data]
dataset_names = ["Non‑living", "Proto‑life", "Life"]

# Store history for plotting
history_scores = []          # list of dicts {frame_name: score}
history_composition = []     # list of dicts {type: count}

for epoch in range(n_epochs):
    # select dataset cyclically
    data_idx = epoch % 3
    data = datasets[data_idx]
    data_name = dataset_names[data_idx]
    
    # For each frame, we need to score it on the entire dataset (10 points).
    # We define score as the sum of individual scores for each point.
    # Our frames' score methods expect a single value, so we loop.
    scores = {}
    for f in controller.frames:
        total_score = 0
        for point in data:
            total_score += f.score(point)
        scores[f.name] = total_score
    
    # Apply reset policy
    survivors, killed = top_k_policy(scores, controller.frames, k=0.3)
    
    # Generate children
    children = []
    for _ in killed:
        parent = np.random.choice(survivors)
        child = agent.propose_child(parent)
        children.append(child)
    
    # Update frames
    controller.frames = survivors + children
    
    # Record history
    history_scores.append(scores)
    comp = {'NonLiving':0, 'ProtoLife':0, 'Life':0}
    for f in controller.frames:
        if isinstance(f, NonLivingFrame):
            comp['NonLiving'] += 1
        elif isinstance(f, ProtoLifeFrame):
            comp['ProtoLife'] += 1
        elif isinstance(f, LifeFrame):
            comp['Life'] += 1
    history_composition.append(comp)
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d} ({data_name:11s}) | "
              f"NonLiving: {comp['NonLiving']:2d} | "
              f"ProtoLife: {comp['ProtoLife']:2d} | "
              f"Life: {comp['Life']:2d}")

## 6. Visualise population evolution


In [ ]:
epochs = range(n_epochs)
nonliving_counts = [h['NonLiving'] for h in history_composition]
protolife_counts = [h['ProtoLife'] for h in history_composition]
life_counts = [h['Life'] for h in history_composition]

plt.figure(figsize=(10,5))
plt.plot(epochs, nonliving_counts, label='Non‑living frames', lw=2)
plt.plot(epochs, protolife_counts, label='Proto‑life frames', lw=2)
plt.plot(epochs, life_counts, label='Life frames', lw=2)

# Shade background by dataset phase (simple alternating)
for i, name in enumerate(dataset_names):
    start = i
    while start < n_epochs:
        end = min(start+1, n_epochs)
        plt.axvspan(start, end, alpha=0.15, color=['gray','lightblue','lightgreen'][i])
        start += 3

plt.xlabel('Epoch')
plt.ylabel('Number of frames')
plt.title('Evolution of GRA frame population')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Show the best frame at the end


In [ ]:
# Evaluate final population on the last dataset (life)
final_scores = {}
for f in controller.frames:
    total = 0
    for point in life_data:
        total += f.score(point)
    final_scores[f.name] = total

best_frame = min(controller.frames, key=lambda f: final_scores[f.name])
print(f"Best frame after evolution: {best_frame.name}")
print(f"Parameters: {best_frame.params}")
print(f"Final score (on life data): {final_scores[best_frame.name]:.3f}")

# Test its predictions on a range of life scores
test_scores = np.linspace(0, 1, 100)
penalties = [best_frame.score(s) for s in test_scores]

plt.figure(figsize=(8,4))
plt.plot(test_scores, penalties)
plt.xlabel('Life score')
plt.ylabel('Penalty (lower is better)')
plt.title(f'Penalty function of best frame: {best_frame.name}')
plt.axvline(life_data.mean(), color='red', ls='--', label='life data mean')
plt.legend()
plt.show()

## 8. Interpretation

- In the beginning (epochs 0‑9), when the system is in the **non‑living** phase, frames that classify low life scores correctly (NonLivingFrame) dominate.
- As the system enters the **proto‑life** phase (epochs 10‑19), the population shifts toward ProtoLifeFrame.
- Finally, in the **life** phase, LifeFrames take over.
- The reset mechanism constantly kills underperforming frames and generates new ones via mutation, allowing the population to track the changing reality.

This demonstrates how GRA obnulënka can be used to model **regime shifts** and **emergent properties** in complex systems.
